# Recurrent Neural Network
### Recommender System

---
This synthetic recommendation dataset simulates user–item interaction sequences for teaching and experimenting with RNN-based next-item prediction models. It contains multiple users, each belonging to a preference cluster (e.g., action, comedy, drama), and their behavior is represented as short sequences of item IDs with smooth transitions within the preferred range. Each sequence is padded to a fixed length, and the goal is to predict the next item the user will interact with. The dataset is split into train and test CSV files, making it lightweight, privacy-safe, and easy to distribute without external downloads.

---

### Libraries

In [1]:
import os
import sys

# Path
sys.path.append(os.path.abspath(".."))

# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn

from numpy import array

import subprocess

from pathlib import Path
import shutil

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.models import *
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l1, l2
from tensorflow.keras import layers

# Model optimization
import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.sparsity.keras import (
    prune,
    pruning_callbacks,
    pruning_schedule
)
from tensorflow_model_optimization.sparsity.keras import strip_pruning, prune_low_magnitude
from tensorflow_model_optimization.sparsity import keras as sparsity

from tensorflow.keras.preprocessing.sequence import pad_sequences


# QKeras (Quantization)
from qkeras import *

# Metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils import shuffle

# Preprocessing
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder

# Dataset split
from sklearn.model_selection import train_test_split

# HLS4ML
import hls4ml

# Reproducibility
tf.random.set_seed(42)

# Clear sessions
K.clear_session()
tf.keras.backend.clear_session()

import sys
sys.path.append(os.path.abspath('../../'))
from common.notebook_utils.distillationClassKeras import Distiller
from common.notebook_utils.utils import  top_k_accuracy, mrr_at_k


os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

2026-03-30 23:01:44.446516: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-30 23:01:44.487750: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-30 23:01:45.189801: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


/tools/anaconda3/envs/neuralEnv/lib/python3.10/site-packages/hls4ml/converters/__init__.py:27: UserWarning: WARNING: Pytorch converter is not enabled!
  warnings.warn("WARNING: Pytorch converter is not enabled!", stacklevel=1)


### Enable GPU 

In [2]:
#  GPU 
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'

import tensorflow as tf
print("GPUs: ", len(tf.config.experimental.list_physical_devices('GPU')))

import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

    except RuntimeError as e:
        print(e)

GPUs:  1


2026-03-30 23:01:46.351605: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-30 23:01:46.399892: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-30 23:01:46.400192: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

### Parameters

In [3]:
num_users=1000
num_items=100
seq_len=30
max_len=30

### Dataset

In [4]:

path = "../00.datasets/"

# Load train sequences and targets
X_train = pd.read_csv(path + "train_sequences.csv").values
y_train = pd.read_csv(path + "train_targets.csv")["target"].values

# Load test sequences and targets
X_test  = pd.read_csv(path + "test_sequences.csv").values
y_test  = pd.read_csv(path + "test_targets.csv")["target"].values

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)


Train: (11600, 30) (11600,)
Test : (2900, 30) (2900,)


---

### Load model

In [5]:
from qkeras.utils import _add_supported_quantized_objects

co = {}
_add_supported_quantized_objects(co)

model = tf.keras.models.load_model('../01.training/models/pmodel_stripped.h5', custom_objects=co)


2026-03-30 23:01:46.525008: I tensorflow/compiler/xla/service/service.cc:169] XLA service 0x1c0dde90 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-03-30 23:01:46.525046: I tensorflow/compiler/xla/service/service.cc:177]   StreamExecutor device (0): Host, Default Version
2026-03-30 23:01:46.525367: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-30 23:01:46.525792: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
20

---

## Integration with a hardware synthesis tool for ML

In [ ]:
# PATH Vitis HLS installation
os.environ['PATH'] = '/tools/Xilinx/Vitis_HLS/2024.1/bin:' + os.environ['PATH']
os.environ['PATH']

'/tools/Xilinx/Xilinx2024/Vitis_HLS/2024.1/bin:/tools/anaconda3/envs/neuralEnv/bin:/tools/anaconda3/condabin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin'

### hls4ml

**hls4ml** is a Python package developed for converting machine learning (ML) models into HLS (High-Level Synthesis) projects, enabling deployment of ML-based inference on hardware like FPGAs. More details can be found at [hls4ml documentation](https://fastmachinelearning.org/hls4ml/).

The user can control several options related to the model, including:

- **Precision:** Define the precision of the calculations in your model (e.g., fixed-point or floating-point representation).

- **Dataflow/Resource Reuse:** Control the level of parallelism or streaming in model implementations, with varying degrees of pipelining.

An HLS configuration should be created using the function `hls4ml.utils.config_from_keras_model(kerasModel, granularity)`, where kerasModel is the pre-trained model you want to implement on an FPGA, and granularity determines the configuration level. The two possible values for granularity are:

- 'model': The same configuration applies to the entire model (e.g., all layers use 16-bit fixed-point precision).

- 'name': Layer-specific configurations can be applied (e.g., the input layer can be defined in 8-bit fixed-point precision, while the second layer is set to 16-bit fixed-point precision).

In [ ]:

# Generate an hls4ml configuration dictionary from the Keras model.
# - granularity='name'         : allows per-layer configuration by layer name
# - default_precision          : sets the default fixed-point format for all layers
#     fixed<16,10> : 16-bit total width, 10 integer bits, 6 fractional bits
hls_config = hls4ml.utils.config_from_keras_model(
    model,
    granularity='name',
    default_precision='fixed<16,10>'
)

# --- Global Model-Level Settings ---

# ReuseFactor=8  : hardware resources are reused 8 times across computations
# RoundingMode   : AP_RND (Round to nearest), more accurate than truncation
# SaturationMode : AP_SAT (Saturate), clamps values to min/max, prevents overflow artifacts
hls_config['Model']['ReuseFactor']    = 8
hls_config['Model']['RoundingMode']   = 'AP_RND'
hls_config['Model']['SaturationMode'] = 'AP_SAT'

# --- Layer-Specific Settings ---

# Use 'Stable' softmax strategy for numerical stability
# Override the global ReuseFactor for this specific layer
hls_config['LayerName']['output_softmax']['Strategy']    = 'Stable'
hls_config['LayerName']['output_softmax']['ReuseFactor'] = 16

Interpreting Model
Topology:
Layer name: input_1, layer type: InputLayer, input shapes: [[None, 30]], output shape: [None, 30]
Layer name: embedding_layer, layer type: Embedding, input shapes: [[None, 30]], output shape: [None, 30, 8]
Layer name: gru_layer, layer type: GRU, input shapes: [[None, 30, 8]], output shape: [None, 16]
Layer name: output_softmax, layer type: Dense, input shapes: [[None, 16]], output shape: [None, 100]


In [ ]:
# Directory path where the HLS (High-Level Synthesis) project will be generated.
# hls4ml translates the neural network into synthesizable RTL code for FPGA deployment.
PATH_HLS_PROJECT = '../02.hls4ml/output/'

# Create configuration for Vitis HLS as backend.
cfg = hls4ml.converters.create_config(backend='Vitis')

cfg['IOType'] = 'io_stream'  
# HLSConfig correspond to the configuration created in hls_config 
cfg['HLSConfig'] = hls_config
# Model to be converted
cfg['KerasModel'] = model
# Folder where the HLS project will be created
cfg['OutputDir'] = PATH_HLS_PROJECT
# FPGA part 
cfg['Part'] = 'xczu3eg-sfvc784-2-e'  



Interpreting Model
Topology:
Layer name: input_1, layer type: InputLayer, input shapes: [[None, 30]], output shape: [None, 30]
Layer name: embedding_layer, layer type: Embedding, input shapes: [[None, 30]], output shape: [None, 30, 8]
Layer name: gru_layer, layer type: GRU, input shapes: [[None, 30, 8]], output shape: [None, 16]
Layer name: output_softmax, layer type: Dense, input shapes: [[None, 16]], output shape: [None, 100]
Creating HLS model
Writing HLS project


Done


bash: /tools/anaconda3/envs/neuralEnv/lib/libtinfo.so.6: no version information available (required by bash)


In [ ]:

# Convert the Keras model to an HLS model using the configuration
hls_model = hls4ml.converters.keras_to_hls(cfg)

# Compile the HLS model for simulation/verification
hls_model.compile()

### Keras vs HLS model evaluation

In [9]:
# --- Prepare Ground Truth Labels ---

# Handle both one-hot encoded labels (2D) and integer labels (1D):
# If y_test is one-hot encoded (e.g. [[0,1,0], [1,0,0]]),
# convert to integer class indices using argmax
if y_test.ndim == 2 and y_test.shape[1] > 1:
    y_true = y_test.argmax(axis=1)
else:
    # If already integer-encoded, just flatten to a 1D array
    y_true = np.array(y_test).ravel()

# Convert X_test to a contiguous float32 array.
X_test_hls = np.ascontiguousarray(X_test, dtype=np.float32)

# --- Run Predictions ---

# Get class probability predictions from the original Keras model
# batch_size=64 processes 64 samples at a time to manage memory
pred_keras = model.predict(X_test, batch_size=64)

# Get class probability predictions from the HLS (FPGA-ready) model
# Uses the contiguous float32 array
pred_hls = hls_model.predict(X_test_hls)

# --- Evaluate Keras Model ---
# Top-K accuracy: checks if the correct class is within the top K predictions
# MRR@K (Mean Reciprocal Rank): measures how highly the correct item is ranked
#         higher MRR → correct recommendation appears earlier in the ranked list
print("=== KERAS  ")
print("Top-1 :", top_k_accuracy(pred_keras, y_true, k=1))   # Exact match rate
print("Top-5 :", top_k_accuracy(pred_keras, y_true, k=5))   # Correct in top 5
print("Top-10:", top_k_accuracy(pred_keras, y_true, k=10))  # Correct in top 10
print("MRR@10:", mrr_at_k(pred_keras, y_true, k=10))        # Ranking quality

# --- Evaluate HLS Model ---
# Same metrics applied to the HLS model to compare against Keras baseline.
# Any drop in accuracy reflects the impact of fixed-point quantization
# and hardware-specific approximations introduced by hls4ml
print("\n=== HLS MODEL")
print("Top-1 :", top_k_accuracy(pred_hls, y_true, k=1))
print("Top-5 :", top_k_accuracy(pred_hls, y_true, k=5))
print("Top-10:", top_k_accuracy(pred_hls, y_true, k=10))
print("MRR@10:", mrr_at_k(pred_hls, y_true, k=10))

46/46 [==============================] - 0s 1ms/step


2026-03-30 23:01:53.096571: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2026-03-30 23:01:53.097869: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2026-03-30 23:01:53.098700: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

=== KERAS  
Top-1 : 0.1910344827586207
Top-5 : 0.646896551724138
Top-10: 0.9279310344827586
MRR@10: 0.3812205801860974

=== HLS MODEL
Top-1 : 0.19586206896551725
Top-5 : 0.5924137931034483
Top-10: 0.9024137931034483
MRR@10: 0.3593433223864258


In [ ]:
# Synthesizes the HLS project and reports key metrics: latency and resource usage.

hls_model.build(csim=False, export=False)

/bin/bash: /tools/anaconda3/envs/neuralEnv/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /tools/anaconda3/envs/neuralEnv/lib/libtinfo.so.6: no version information available (required by /bin/bash)


/bin/bash: /tools/anaconda3/envs/neuralEnv/lib/libtinfo.so.6: no version information available (required by /bin/bash)



****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2024.1 (64-bit)
  **** SW Build 5069499 on May 21 2024
  **** IP Build 5075265 on Wed May 22 21:45:21 MDT 2024
  **** SharedData Build 5076995 on Wed May 22 18:29:18 MDT 2024
  **** Start of session at: Mon Mar 30 23:01:56 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2024 Advanced Micro Devices, Inc. All Rights Reserved.

source /tools/Xilinx/Xilinx2024/Vitis_HLS/2024.1/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 200-10] For user 'ro' on host 'mareKaleido' (Linux_x86_64 version 5.15.0-139-generic) on Mon Mar 30 23:01:58 CEST 2026
INFO: [HLS 200-10] On os Ubuntu 20.04.6 LTS
INFO: [HLS 200-10] In directory '/home/ro/kaleido/repo/github/aup-zu3-amd-ml/03-rnn/02.hls4ml/output'
Sourcing Tcl script 'build_prj.tcl'
INFO: [HLS 200-1510] Running: open_project myproject_prj 
INFO: [HLS 200-10] Creating and opening project '/home/ro/kaleido/repo/github/aup-zu3-amd-ml/03-rnn/02.hls4ml/outp

{'CSynthesisReport': {'TargetClockPeriod': '5.00',
  'EstimatedClockPeriod': '4.262',
  'BestLatency': '271',
  'WorstLatency': '271',
  'IntervalMin': '260',
  'IntervalMax': '260',
  'BRAM_18K': '82',
  'DSP': '265',
  'FF': '27194',
  'LUT': '161710',
  'URAM': '0',
  'AvailableBRAM_18K': '432',
  'AvailableDSP': '360',
  'AvailableFF': '141120',
  'AvailableLUT': '70560',
  'AvailableURAM': '0'}}

Once the hls4ml project has been synthesized and the hardware metrics have been reviewed (latency, throughput, LUT/FF/DSP/BRAM utilization), the next step is to enable FPGA integration.

To achieve this, an AXI-based wrapper must be added around the generated HLS model, typically including AXI4-Stream interfaces for data movement and AXI-Lite for control.

This wrapper allows the design to be driven by a DMA engine or other AXI infrastructure in the FPGA system, enabling high-performance inference from the processing system or external data sources.

In [ ]:
# Add AXI-Stream support

# Copy the .cpp and .h files that contain the AXI-based wrapper into the firmware/ directory of the hls4ml project.
# These files provide the AXI4-Stream and AXI-Lite interfaces required for DMA-driven inference on the FPGA.

shutil.copy("../03.hls/myproject_embedding_accel.cpp", os.path.join(PATH_HLS_PROJECT, "firmware"))
shutil.copy("../03.hls/myproject_embedding_accel.h", os.path.join(PATH_HLS_PROJECT, "firmware"))

'../02.hls4ml/output/firmware/myproject_embedding_accel.h'

The *build_accel.tcl* script defines the complete HLS build process: itcreates a clean Vitis HLS project, adds the hls4ml sources plus the AXI/DMA wrapper, configures the FPGA part, sets the clock, and triggers synthesis. Also,  it generates a fully packaged IP core that can be directly imported into Vivado.

In [12]:
# Copy tcl file 
shutil.copy("../03.hls/build_accel.tcl", PATH_HLS_PROJECT)

'../02.hls4ml/output/build_accel.tcl'

The TCL script is executed from Python using subprocess.run.

This produces a fully synthesized IP block inside:  \<project\>/myproject_accel_prj/solution1/impl/ip/*


In [13]:
# Execute tcl file to generate the corresponig HLS project + IP core

PROJ_DIR = Path(PATH_HLS_PROJECT)

res = subprocess.run(
    ["vitis_hls", "-f", "build_accel.tcl"],
    cwd=PROJ_DIR,                      
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Return code:", res.returncode)
print(res.stdout)


Return code: 0
/bin/bash: /tools/anaconda3/envs/neuralEnv/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /tools/anaconda3/envs/neuralEnv/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /tools/anaconda3/envs/neuralEnv/lib/libtinfo.so.6: no version information available (required by /bin/bash)

****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2024.1 (64-bit)
  **** SW Build 5069499 on May 21 2024
  **** IP Build 5075265 on Wed May 22 21:45:21 MDT 2024
  **** SharedData Build 5076995 on Wed May 22 18:29:18 MDT 2024
  **** Start of session at: Mon Mar 30 23:03:58 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2024 Advanced Micro Devices, Inc. All Rights Reserved.

source /tools/Xilinx/Xilinx2024/Vitis_HLS/2024.1/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 200-10] For user 'ro' on host 'mareKaleido' (Linux_x86_64 version 5.15.0-139-generic) on Mon Mar 30 2

From this point, once the IP core has been generated, you can create the corresponding hardware in Vivado, which will ultimately be used in the PYNQ framework.

Open Vivado and follow the steps in the wiki to continue the workflow.


---

This work was supported in part by the [AMD University Program](https://www.amd.com/en/corporate/university-program.html) 